# Cobalt L-edge energy sweep: universal phase retrieval

Jointly reconstruct the fixed magnetic state from ideal CR (`+1`) and CL (`-1`) holograms at 20 energies. Every hologram is stretched about the detector center by $E/E_{\min}$ and center-cropped back to the original array size. Phase retrieval uses only the support mask stored at $E_{\min}$. The stored `xmcd_logs` are loaded only at the end for validation.

In [ ]:
from pathlib import Path
import sys

import h5py
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage

# Make imports work when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "library").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from library import phase_retrieval_universal as pr

DATA_FILE = REPO_ROOT / "Data" / "cobalt_l_edge_energy_sweep.h5"

if not DATA_FILE.exists():
    raise FileNotFoundError(DATA_FILE)


In [ ]:

DATA_FILE= "/Users/riccardo/Data/cobalt_l_edge_energy_sweep.h5"

plt.rcParams.update({"figure.figsize": (7, 5), "image.cmap": "gray"})
print(DATA_FILE)

## Load the ideal holograms

Each energy group contains arrays with shape `(1, 512, 512)`. Squeezing and stacking produces one `(n_energy, ny, nx)` intensity stack per polarization.

In [ ]:
load_hologram="detected_holograms"


with h5py.File(DATA_FILE, "r") as handle:
    energies_eV = np.asarray(handle["energies"], dtype=float)
    group_names = sorted(handle[load_hologram].keys())

    if len(group_names) != len(energies_eV):
        raise ValueError("Number of ideal-hologram groups does not match energies")

    cr_ideal = np.stack([
        np.squeeze(np.asarray(handle[f"{load_hologram}/{name}/CR"], dtype=float))
        for name in group_names
    ])
    cl_ideal = np.stack([
        np.squeeze(np.asarray(handle[f"{load_hologram}/{name}/CL"], dtype=float))
        for name in group_names
    ])

    emin_index = int(np.argmin(energies_eV))
    emin_group = group_names[emin_index]
    supportmask = np.asarray(
        handle[f"supportmasks/{emin_group}"], dtype=float
    )

if cr_ideal.shape != cl_ideal.shape:
    raise ValueError(f"CR and CL shapes differ: {cr_ideal.shape} vs {cl_ideal.shape}")
if cr_ideal.shape[0] != len(energies_eV):
    raise ValueError("The hologram energy axis does not match energies")
if np.any(~np.isfinite(cr_ideal)) or np.any(~np.isfinite(cl_ideal)):
    raise ValueError("The ideal holograms contain NaN or infinite values")
if np.min(cr_ideal) < 0 or np.min(cl_ideal) < 0:
    raise ValueError("The ideal holograms must be non-negative intensities")

n_energy, ny, nx = cr_ideal.shape
print(f"energies: {energies_eV[0]:.1f} to {energies_eV[-1]:.1f} eV ({n_energy} points)")
print("CR stack:", cr_ideal.shape, cr_ideal.dtype)
print("CL stack:", cl_ideal.shape, cl_ideal.dtype)
print(f"Emin support: {emin_group} at {energies_eV[emin_index]:.1f} eV")

In [ ]:

with h5py.File(DATA_FILE, "r") as handle:
    mask_pixel=(np.squeeze(np.asarray(handle[f"detected_holograms_without_beamstop/00000/CR"], dtype=float)))-(np.squeeze(np.asarray(handle[f"detected_holograms/00000/CR"], dtype=float)))

In [ ]:
fig,ax=plt.subplots()
ax.imshow(mask_pixel>1e-3)

In [ ]:
%matplotlib widget

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(15, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(stack[index])))))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram intensities, log display")
plt.tight_layout()

In [ ]:
%matplotlib widget

show_indices = np.unique(np.linspace(0, n_energy - 1, 3, dtype=int))
fig, axes = plt.subplots(2, len(show_indices), figsize=(15, 6), squeeze=False, sharex=True, sharey=True)
for column, index in enumerate(show_indices):
    for row, (stack, label) in enumerate(((cr_ideal, "CR"), (cl_ideal, "CL"))):
        axes[row, column].imshow(np.log1p(stack[index]))
        axes[row, column].set_title(f"{label}, {energies_eV[index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Ideal hologram intensities, log display")
plt.tight_layout()

## Normalize detector sampling and build metadata

The reciprocal-space scale changes with photon energy. For each energy $E$, the detector image is stretched about its center by $E/E_{\min}$ and sampled directly onto the original `(512, 512)` grid. This is equivalent to stretching followed by a centered crop, while avoiding a temporary larger array.

Only `supportmasks/<Emin index>` is supplied to phase retrieval. For ideal holograms all output detector pixels are measured: `mask_pixel == 0`.

In [ ]:
def centered_rescale(image, scale, output_shape=None, order=1):
    """Rescale about the array center and return a fixed-size center crop."""
    image = np.asarray(image)
    if image.ndim != 2:
        raise ValueError(f"Expected a 2D image, got shape {image.shape}")
    if scale <= 0 or not np.isfinite(scale):
        raise ValueError("scale must be positive and finite")
    if output_shape is None:
        output_shape = image.shape
    if np.isclose(scale, 1.0) and tuple(output_shape) == image.shape:
        return image.copy()

    input_center = (np.asarray(image.shape, dtype=float) - 1.0) / 2.0
    output_center = (np.asarray(output_shape, dtype=float) - 1.0) / 2.0
    matrix = np.eye(2) / float(scale)
    offset = input_center - matrix @ output_center
    return ndimage.affine_transform(
        image,
        matrix=matrix,
        offset=offset,
        output_shape=tuple(output_shape),
        order=order,
        mode="constant",
        cval=0.0,
        prefilter=False,
    )

emin_eV = float(energies_eV[emin_index])
scale_factors = energies_eV / emin_eV
cr_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=1)
    for image, scale in zip(cr_ideal, scale_factors)
])
cl_rescaled = np.stack([
    centered_rescale(image, scale, output_shape=(ny, nx), order=1)
    for image, scale in zip(cl_ideal, scale_factors)
])

# Linear interpolation of non-negative inputs should remain non-negative;
# clip tiny floating-point undershoots defensively.
cr_rescaled = np.maximum(cr_rescaled, 0.0)
cl_rescaled = np.maximum(cl_rescaled, 0.0)
mask_pixel = np.zeros((ny, nx), dtype=np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(supportmask)
axes[0].set_title(f"Only support used: {emin_eV:.1f} eV")
axes[1].imshow(np.log1p(cr_ideal[-1]))
axes[1].set_title(f"Raw CR at {energies_eV[-1]:.1f} eV")
axes[2].imshow(np.log1p(cr_rescaled[-1]))
axes[2].set_title(f"Stretched by {scale_factors[-1]:.4f} and cropped")
for axis in axes:
    axis.axis("off")
plt.tight_layout()

print(f"Emin support pixels: {int(supportmask.sum())} / {supportmask.size}")
print(f"Scale-factor range: {scale_factors.min():.6f} to {scale_factors.max():.6f}")
print("Rescaled stacks:", cr_rescaled.shape, cl_rescaled.shape)

In [ ]:
# Interleave CR and CL so each adjacent pair belongs to the same energy.
holograms = np.stack([
    image
    for energy_index in range(n_energy)
    for image in (cr_rescaled[energy_index], cl_rescaled[energy_index])
])

state_labels = ["fixed_state"] * (2 * n_energy)
energy_labels = np.repeat(energies_eV, 2)
polarizations = np.tile([+1.0, -1.0], n_energy)  # CR positive, CL negative
illumination_labels = ["fixed_beam"] * (2 * n_energy)

print("joint hologram stack:", holograms.shape)
print("first four (energy, polarization):", list(zip(energy_labels, polarizations))[:4])

## Joint phase retrieval

`QUICK_RUN=True` is a short end-to-end check. Set it to `False` for the longer starting recipe, then tune the iteration counts based on convergence. The physical model is

$$L(E,p)=C+q_c(E)+p\,q_m(E)m_z,$$

with one shared state and beam. No ground-truth arrays are passed to the reconstruction.

In [ ]:
QUICK_RUN = False

if QUICK_RUN:
    inner_Nit = [5, 1]
    outer_iterations = 5
    warmup_Nit = [20, 5]
    physical_iterations = 3
else:
    inner_Nit = [200, 50]
    outer_iterations = 2
    warmup_Nit = [200, 50]
    physical_iterations = 5

recipe = {
    "projection_model": "physical_factorized",
    "inner_mode": ["HAPRE", "ER"],
    "inner_Nit": inner_Nit,
    "outer_iterations": outer_iterations,
    "warmup_mode": ["HAPRE", "ER"],
    "warmup_Nit": warmup_Nit,
    "beta_zero": 0.5,
    "beta_mode": "arctan",
    "alpha_zero": 0.0,
    "TV_freq": 1e9,
    "average_img": 1,
    "plot_every": 20,
    "shuffle_observations": True,
    "random_seed": 7,
    "projection_every": 1,
    "projection_start": 0,
    "projection_relaxation": 1.0,
    "physical_iterations": physical_iterations,
    "energy_values": energies_eV,
    "charge_spectral_constraint": "free",
    "magnetic_spectral_constraint": "free",
    "final_fourier_constraint": True,
}

fields, components, bsmasks, errors = pr.universal_phase_retrieval_algorithm(
    holograms,
    mask_pixel,
    supportmask,
    state_labels=state_labels,
    energy_labels=energy_labels,
    polarization_coefficients=polarizations,
    illumination_labels=illumination_labels,
    saturated_states=None,
    universal_recipe=recipe,
)

print(f"fit residual RMS: {components['fit_residual_rms']:.4g}")
print("magnetic scale anchored:", components["magnetic_scale_anchored"])
print("fully identifiable:", components["identifiable"])
print(f"runtime: {errors['runtime_seconds']:.1f} s")

With one unknown state, the product $q_m(E)m_z$ is identifiable but its factors have a scale/sign ambiguity. The library clips the recovered reduced magnetization to `[-1, 1]`. A known saturated state or an absolutely calibrated magnetic spectrum is required to anchor the physical scale.

In [ ]:
magnetization = components["magnetization_by_state"]["fixed_state"]
charge_response = np.asarray(components["charge_response"])
magnetic_response = np.asarray(components["magnetic_response"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
image = axes[0].imshow(magnetization, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0].set_title("Recovered magnetic state $m_z$")
axes[0].axis("off")
fig.colorbar(image, ax=axes[0], fraction=0.046)

axes[1].plot(energies_eV, charge_response.real, "o-", label="Re")
axes[1].plot(energies_eV, charge_response.imag, "o-", label="Im")
axes[1].set_title("Recovered charge response")
axes[1].set_xlabel("Energy (eV)")
axes[1].legend()

axes[2].plot(energies_eV, magnetic_response.real, "o-", label="Re")
axes[2].plot(energies_eV, magnetic_response.imag, "o-", label="Im")
axes[2].set_title("Recovered magnetic response")
axes[2].set_xlabel("Energy (eV)")
axes[2].legend()
plt.tight_layout()

## Validate against the simulated XMCD state

The complex `xmcd_logs` obey `log(CR/CL) = 2 q_m(E)m_z`. Because stretching reciprocal-space data by $E/E_{\min}$ contracts the corresponding real-space coordinates by $E_{\min}/E$, each validation map is normalized by that inverse factor before the rank-one decomposition. This validation data was not used by phase retrieval.

In [ ]:
with h5py.File(DATA_FILE, "r") as handle:
    true_xmcd_logs = np.stack([
        np.squeeze(np.asarray(handle[f"xmcd_logs/{name}"]))
        for name in group_names
    ])

true_xmcd_logs = np.stack([
    centered_rescale(log_map, 1.0 / scale, output_shape=(ny, nx), order=1)
    for log_map, scale in zip(true_xmcd_logs, scale_factors)
])

# The first spatial right-singular vector is the common state times an
# arbitrary complex phase. Rotate it to be maximally real, then normalize.
_, _, vh = np.linalg.svd(true_xmcd_logs.reshape(n_energy, -1), full_matrices=False)
truth_complex = vh[0].reshape(ny, nx)
truth_phase = 0.5 * np.angle(np.sum(truth_complex ** 2))
truth_state = np.real(truth_complex * np.exp(-1j * truth_phase))
truth_state /= np.max(np.abs(truth_state))

# Resolve the arbitrary global sign for comparison only.
inside = supportmask.astype(bool)
if np.sum(magnetization[inside] * truth_state[inside]) < 0:
    truth_state *= -1

correlation = np.corrcoef(magnetization[inside], truth_state[inside])[0, 1]
difference = magnetization - truth_state

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for axis, image_data, title in zip(
    axes,
    (magnetization, truth_state, difference),
    ("Phase-retrieval state", "Simulated XMCD state", "Difference"),
):
    image = axis.imshow(image_data, cmap="RdBu_r", vmin=-1, vmax=1)
    axis.set_title(title)
    axis.axis("off")
    fig.colorbar(image, ax=axis, fraction=0.046)
plt.suptitle(f"State correlation inside support: {correlation:.4f}")
plt.tight_layout()

print(f"state correlation inside support: {correlation:.6f}")

In [ ]:
%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fields)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(15, 6), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.angle(exit_waves[2 * energy_index + offset])
        axes[row, column].imshow(phase, cmap="twilight", vmin=-np.pi, vmax=np.pi)
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

In [ ]:
%matplotlib widget
# Inspect reconstructed real-space exit-wave phases at representative energies.
object_logs = pr.fourier_field_to_object_log(fields)
exit_waves = np.exp(object_logs)

fig, axes = plt.subplots(2, len(show_indices), figsize=(15, 6), squeeze=False)
for column, energy_index in enumerate(show_indices):
    for row, (offset, label) in enumerate(((0, "CR"), (1, "CL"))):
        phase = np.abs(exit_waves[2 * energy_index + 1] - exit_waves[2 * energy_index + 0])
        axes[row, column].imshow(np.fft.fftshift(phase)[220:-220, 220:-220])
        axes[row, column].set_title(f"{label}, {energies_eV[energy_index]:.1f} eV")
        axes[row, column].axis("off")
plt.suptitle("Reconstructed exit-wave phase")
plt.tight_layout()

In [ ]:
components["charge_spectral_info"]

In [ ]:
components.keys()

In [ ]:
components["charge_response"]

In [ ]:
fig,ax=plt.subplots(2,2)
ax[0,0].plot(np.abs(components["charge_response"]))
ax[0,1].plot(np.abs(components["magnetic_response"]))
ax[1,0].plot(np.angle(components["charge_response"]))
ax[1,1].plot(np.angle(components["magnetic_response"]))

In [ ]:
fig,ax=plt.subplots()
ax.imshow(np.fft.fftshift(np.real(components["magnetization"][0]))[220:-220, 220:-220], vmin=-0.006, vmax=0.006)